In [1]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110
print("project root:", _root)

project root: /workspaces/geo_distance_arena


# 03 - The Detour Factor

The gap between road and straight-line distance, measured across the city. The ratio (circuity, or detour factor) averages ~1.3-1.4 but has a long, fat tail at the barriers. Because the variance is large and spatially structured, **a single average multiplier cannot fix straight-line distance** -- the barrier cases stay wrong.

In [2]:
import random
import numpy as np
import matplotlib.pyplot as plt

from src.graph.build import load_city_graph
from src.graph.snap import SnapIndex
from src.evaluation.detour_factor import detour_factors, summarize
from src.distance.haversine import haversine_m
from src.distance.road import road_distance_m
from src.scenarios.points import load_scenario
from src.visualisation import detour_factor_histogram
from src.io_utils import save_fig, save_json
import networkx as nx

drive = load_city_graph('copenhagen', 'drive')
scenario = load_scenario('copenhagen')
depot = tuple(scenario['depot'])
customers = [tuple(c) for c in scenario['customers']]

## The distribution of the detour factor

Many random origin-destination node pairs.

In [3]:
rng = random.Random(42)
nodes = list(drive.nodes)
node_pairs = []
while len(node_pairs) < 400:
    a, b = rng.choice(nodes), rng.choice(nodes)
    if a != b and nx.has_path(drive, a, b):
        node_pairs.append((a, b))
factors = detour_factors(drive, node_pairs)
stats = summarize(factors)
save_json(stats, '03_detour_summary')
stats

{'mean': 1.3671219100864807,
 'median': 1.2868339913787823,
 'p90': 1.6123412542599276,
 'max': 3.373983911895483,
 'std': 0.30469568871254804}

In [4]:
fig = detour_factor_histogram(factors)
save_fig(fig, '03_detour_histogram')
import matplotlib.pyplot as plt
plt.show()

## Where the detour explodes: map the ratio

Depot-to-customer detour factor, each customer coloured by its ratio. The barriers (the rivers) light up.

In [5]:
index = SnapIndex.from_graph(drive)
depot_node, _ = index.nearest(*depot)
rows = []
for lat, lon in customers:
    node, _ = index.nearest(lat, lon)
    straight = haversine_m(depot[0], depot[1], lat, lon)
    if straight <= 0 or not nx.has_path(drive, depot_node, node):
        continue
    rows.append((lat, lon, road_distance_m(drive, depot_node, node) / straight))
clat = [r[0] for r in rows]
clon = [r[1] for r in rows]
cfac = [r[2] for r in rows]
save_json({'lat': clat, 'lon': clon, 'detour_factor': cfac}, '03_detour_map_points')
import osmnx as ox
fig, ax = ox.plot_graph(drive, node_size=0, edge_color='#dddddd', edge_linewidth=0.5,
                        bgcolor='white', show=False, close=False)
sc = ax.scatter(clon, clat, c=cfac, cmap='inferno', s=28, vmin=1.0, vmax=3.0, zorder=3)
ax.scatter([depot[1]], [depot[0]], marker='*', c='#1f77b4', s=220, zorder=4, label='depot')
fig.colorbar(sc, ax=ax, label='road / straight (detour factor)')
ax.set_title('Detour factor from the depot: barriers light up')
ax.legend(loc='upper right')
save_fig(fig, '03_detour_map')
plt.show()

## A single multiplier does not fix it

Correct every straight-line distance by the mean detour factor, then look at the residual error. The average is fixed by construction, but the barrier cases are still wrong by tens of percent.

In [6]:
factors_arr = np.array(cfac)
mean_factor = float(factors_arr.mean())
residual_pct = (mean_factor - factors_arr) / factors_arr * 100
resid_stats = {
    'mean_factor': mean_factor,
    'residual_pct_p05': float(np.percentile(residual_pct, 5)),
    'residual_pct_p95': float(np.percentile(residual_pct, 95)),
    'residual_pct_abs_max': float(np.abs(residual_pct).max()),
}
save_json(resid_stats, '03_single_multiplier_residual')
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(residual_pct, bins=30, color='#72B7B2', edgecolor='white')
ax.axvline(0, color='#E45756', ls='--', label='perfect after correction')
ax.set_xlabel('residual error after a single-multiplier correction (%)')
ax.set_ylabel('count')
ax.set_title('One multiplier fixes the average, not the barriers')
ax.legend()
fig.tight_layout()
save_fig(fig, '03_single_multiplier_residual')
plt.show()

**Takeaway.** The detour factor is not a constant. Its mean is useful context, but its spread -- driven by rivers and highways -- means the only reliable fix is the real network, not a correction factor.